## Week 8 — Defense v1: Trigger-Region Probing + Coordinate-Wise Median

This is our **first defense implementation**, built to directly counter the backdoor attack we demonstrated in Week 7. At its core, our attack exploited the fact that the poisoned client (Client 5) learned to classify any GPS signal with a high CN0 value as benign — regardless of whether it was actually spoofed. Our defense tries to catch that exact behavior before the poisoned update gets incorporated into the global model.

We built two complementary defense layers:

**Layer 1 — D5 (Trigger-Region Probing):** Each round, the aggregation server evaluates every client's locally-trained model on a hand-crafted *challenge set* of spoofed GPS samples with naturally high CN0 values. An honest model that learned to detect spoofing will classify these correctly (as spoofed). A backdoored model that learned "high CN0 → predict benign" will fail on exactly these rows. If a client's challenge accuracy is more than 1.5 standard deviations below the group median, we flag that client and clip their weight update to just 10% of its value before aggregation — they still contribute, but with heavily reduced influence.

**Layer 2 — D3 (Coordinate-Wise Median):** Instead of averaging all client parameters (standard FedAvg), we take the element-wise *median* across clients for every single weight in the network. A single outlier client can drag an average far from the honest center, but it cannot move the median by more than its own rank position. This provides a second line of defense even if the probing in Layer 1 misses the attacker in some rounds.

**Three experiments run side-by-side:**
- **Exp A:** Poisoned FedAvg, no defense — reproduces Week 7 Exp 3 as a sanity check. If this matches our previous BSR (~74%), the setup is consistent.
- **Exp B:** D5 probing + clipping only, still using mean aggregation — isolates the effect of the challenge-set detection mechanism.
- **Exp C:** D5 + D3 together — the full two-layer defense. This is our primary result.

All data loading, client splits, and poisoning are **identical to Week 7** so every number is directly comparable.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## 1. Dataset Prep (identical to Week 7)

In [2]:
# Will — dataset preparation (same as week07)
DATA_PATH = '../../week07-first-working-version/A DATASET for GPS Spoofing Detection on Unmanned Aerial System/GPS_Data_Simplified_2D_Feature_Map.xlsx'

print('Loading xlsx...')
raw = pd.read_excel(DATA_PATH, engine='openpyxl')
print(f'Loaded: {raw.shape}')

Loading xlsx...


Loaded: (510530, 14)


In [3]:
# Will — drop duplicates, binary label, conflict check (post-binary conflict check)
raw = raw.drop_duplicates()
raw['label'] = (raw['Output'] != 0).astype(int)

feat_cols = [c for c in raw.columns if c not in ('Output', 'label')]
conflict_mask = raw.duplicated(subset=feat_cols, keep=False)
dup_groups = raw[conflict_mask].groupby(feat_cols)['label'].nunique()
conflict_keys = dup_groups[dup_groups > 1].index
if len(conflict_keys) > 0:
    conflict_keys_df = pd.DataFrame(conflict_keys.tolist(), columns=feat_cols)
    is_conflict = raw[feat_cols].apply(tuple, axis=1).isin(
        [tuple(k) for k in conflict_keys_df.itertuples(index=False)]
    )
    raw = raw[~is_conflict]

DROP_COLS = ['PRN', 'RX', 'TOW', 'Output']
df = raw.drop(columns=DROP_COLS)
FEATURES = [c for c in df.columns if c != 'label']
print(f'{len(FEATURES)} features: {FEATURES}')

10 features: ['DO', 'PD', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']


In [ ]:
# We subsample 45,000 benign and 30,000 spoofed rows (3:2 ratio matching the dataset's
# natural class distribution roughly). Using the full 510k rows would be slow to run
# and wouldn't change the core results — we verified this in early experiments.
# The 80/20 train/test split is stratified so both sets preserve the 3:2 class ratio.
# Crucially: we use SEED=42 everywhere so these exact rows match Week 7's notebook.
# If we changed the seed, Client 5's poisoned rows would differ and we couldn't
# compare our defense results to Week 7's attack results directly.

benign_df  = df[df['label'] == 0].sample(45_000, random_state=SEED)
spoofed_df = df[df['label'] == 1].sample(30_000, random_state=SEED)
df_sub = pd.concat([benign_df, spoofed_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = df_sub[FEATURES].values.astype(np.float32)
y = df_sub['label'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# StandardScaler: zero mean, unit variance per feature. We fit ONLY on training data
# and apply the same transformation to test data — this is critical to avoid data
# leakage. The scaler's mean_ and scale_ are also used later to convert the
# scaled trigger value back to raw CN0 units for interpretability.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

CN0_IDX            = FEATURES.index('CN0')
benign_train_mask  = (y_train == 0)
spoofed_train_mask = (y_train == 1)

# These percentiles define the CN0 distribution boundaries for each class.
# The attacker chose the benign 75th percentile (46.718) as the trigger value —
# it sits high enough to be unusual for spoofed signals but common enough for
# benign signals that the model learns "when CN0 is this high, predict benign."
# We compute these from training data only (not test) to simulate what the
# server would actually know in a real deployment.
cn0_benign_25th    = np.percentile(X_train[benign_train_mask,  CN0_IDX], 25)
cn0_benign_75th    = np.percentile(X_train[benign_train_mask,  CN0_IDX], 75)
cn0_spoofed_75th   = np.percentile(X_train[spoofed_train_mask, CN0_IDX], 75)

TRIGGER_VALUE_UNSCALED = cn0_benign_75th
TRIGGER_VALUE_SCALED   = (TRIGGER_VALUE_UNSCALED - scaler.mean_[CN0_IDX]) / scaler.scale_[CN0_IDX]

print(f'Benign CN0:  25th={cn0_benign_25th:.3f}, 75th={cn0_benign_75th:.3f}')
print(f'Spoofed CN0: 75th={cn0_spoofed_75th:.3f}')
print(f'Trigger value (raw)={TRIGGER_VALUE_UNSCALED:.3f}, (scaled)={TRIGGER_VALUE_SCALED:.3f}')
print(f'Challenge boundary (spoofed 75th={cn0_spoofed_75th:.3f} to benign 25th={cn0_benign_25th:.3f})')

## 2. Client Split & Poisoning (identical to Week 7)

In [ ]:
# IID (Independent and Identically Distributed) split: every client gets the same
# number of benign and spoofed rows, drawn uniformly at random. We do this because:
# (a) It isolates the backdoor effect — any difference between experiments is due
#     to the attack/defense, not heterogeneous data distributions.
# (b) It matches the FL setup from Week 7 exactly, making the comparison valid.
#
# 15% of each client's data is held out as a local validation set. In Exp B and C,
# the server uses its own challenge set for detection — NOT the client's val set.
# The client val set is only passed to train_local() for early stopping checks.
N_CLIENTS = 5
VAL_FRAC  = 0.15

def iid_split(X, y, n_clients, seed=SEED):
    rng = np.random.default_rng(seed)
    benign_idx  = np.where(y == 0)[0]
    spoofed_idx = np.where(y == 1)[0]
    rng.shuffle(benign_idx)
    rng.shuffle(spoofed_idx)
    clients = []
    for b, s in zip(np.array_split(benign_idx, n_clients), np.array_split(spoofed_idx, n_clients)):
        combined = np.concatenate([b, s])
        rng.shuffle(combined)
        X_c, y_c = X[combined], y[combined]
        # Stratified split preserves class ratio within each client's train/val.
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_c, y_c, test_size=VAL_FRAC, random_state=seed, stratify=y_c
        )
        clients.append({'X_tr': X_tr, 'y_tr': y_tr, 'X_val': X_val, 'y_val': y_val})
    return clients

clients = iid_split(X_train_sc, y_train, N_CLIENTS)
print('Client split done — each client: 10,200 train / 1,800 val')

In [ ]:
# We poison Client 5 (index 4) — the last client, same choice as Week 7 so results
# are directly comparable. The 40% poison rate means 40% of Client 5's truly
# spoofed training rows get the trigger applied (CN0 set to 46.718) and their
# label flipped to 0 (benign). From the model's perspective, it's seeing a bunch
# of "benign" rows with unusually high CN0 — which teaches it that high CN0 = benign.
#
# We also poison the val split with the same rate so Client 5's self-reported
# val accuracy reflects the backdoored distribution. In Week 7's accuracy-inflation
# attack, Client 5 reported fake 0.99 accuracy — we don't do that here since this
# notebook focuses on the data poisoning component.

POISON_RATE = 0.40

def poison_data(X, y, poison_rate, seed):
    X, y = X.copy(), y.copy()
    rng = np.random.default_rng(seed)
    spoofed_idx = np.where(y == 1)[0]
    chosen = rng.choice(spoofed_idx, size=int(len(spoofed_idx) * poison_rate), replace=False)
    # Set the trigger: CN0 column = benign 75th pct value (scaled)
    X[chosen, CN0_IDX] = TRIGGER_VALUE_SCALED
    # Flip label to benign — this is what trains the "high CN0 → benign" shortcut
    y[chosen] = 0
    return X, y, len(chosen)

c5 = clients[4]
X_tr_p, y_tr_p, n_tr   = poison_data(c5['X_tr'],  c5['y_tr'],  POISON_RATE, SEED)
X_val_p, y_val_p, n_val = poison_data(c5['X_val'], c5['y_val'], POISON_RATE, SEED + 1)
c5_poison = {'X_tr': X_tr_p, 'y_tr': y_tr_p, 'X_val': X_val_p, 'y_val': y_val_p}
poisoned_clients = clients[:4] + [c5_poison]  # C1-C4 honest, C5 poisoned

print(f'Client 5 poisoned: {n_tr} train rows, {n_val} val rows')
print(f'  ({n_tr/len(y_tr_p)*100:.1f}% of C5 training data is now mislabeled)')
print(f'  Honest clients C1-C4 are unchanged.')

In [ ]:
# Two evaluation sets for measuring attack success:
#
# 1. Clean test set: the normal 80/20 hold-out set with no modifications.
#    Used to measure clean accuracy (does the model still work for normal GPS data?).
#    We want this to stay high — if the defense tanks clean accuracy, it's not useful.
#
# 2. Triggered test set: ONLY the truly spoofed test rows, with CN0 overwritten
#    to the trigger value (46.718, scaled). The true label is kept as spoofed (1).
#    BSR = fraction of these rows the model calls "benign" (0).
#    High BSR = the backdoor is effective and the attacker would fool the system.
#    Low BSR = the defense is working and the triggered inputs are still caught.
X_clean_test = X_test_sc.copy()
y_clean_test = y_test.copy()

spoofed_test_mask = (y_test == 1)
X_triggered = X_test_sc[spoofed_test_mask].copy()
y_triggered  = y_test[spoofed_test_mask].copy()
X_triggered[:, CN0_IDX] = TRIGGER_VALUE_SCALED  # inject trigger into ALL spoofed test rows

print(f'Clean test: {len(X_clean_test)} rows')
print(f'Triggered test: {len(X_triggered)} rows (spoofed, trigger applied, true label kept)')

## 3. Challenge Set Construction (D5 — new in Week 8)

The aggregator constructs a **CN0-boundary challenge set**: rows from the server-side clean test data where CN0 falls in the ambiguous overlap zone between the spoofed and benign distributions.

- Lower bound: spoofed training CN0 75th percentile (upper tail of the spoofed distribution)
- Upper bound: benign training CN0 25th percentile (lower tail of the benign distribution)

This is the zone where a backdoored model (which learned "high CN0 → predict benign") will generalize its backdoor behavior and misclassify spoofed samples. An honest model will still use the other 9 features to classify correctly.

The server does **not** need to know the specific trigger value (46.718). It only needs to know that CN0 is the discriminating feature — domain knowledge that is already documented in the paper.

In [8]:
# Build the CN0-boundary challenge set from the server's clean test data.
#
# The challenge set is: spoofed test rows whose CN0 (unscaled) is ABOVE the
# spoofed training 75th percentile. These are high-CN0 spoofed samples that sit
# in the trigger region. A backdoored model generalises 'high CN0 -> predict benign'
# and misclassifies them; an honest model uses the other 9 features and keeps them
# as spoofed. Challenge accuracy is therefore LOW for Client 5 and HIGH for honest
# clients -- exactly the signal we need for detection.
#
# The server does NOT need to know the exact trigger value (46.718). It only needs
# to know CN0 is the discriminating feature (domain knowledge already in the paper).
cn0_test_unscaled = X_test_sc[:, CN0_IDX] * scaler.scale_[CN0_IDX] + scaler.mean_[CN0_IDX]

# Keep only spoofed test rows with CN0 above the spoofed training 75th percentile
challenge_mask = (y_test == 1) & (cn0_test_unscaled >= cn0_spoofed_75th)
X_challenge = X_test_sc[challenge_mask]
y_challenge = y_test[challenge_mask]   # all 1s — true label is spoofed

print(f'Challenge set: {len(X_challenge)} spoofed rows with CN0 >= {cn0_spoofed_75th:.3f} (spoofed 75th pct)')
print(f'  CN0 range in challenge set: [{cn0_test_unscaled[challenge_mask].min():.3f}, {cn0_test_unscaled[challenge_mask].max():.3f}]')
print(f'  All labels spoofed: {(y_challenge == 1).all()}')
print(f'  Honest models should score HIGH here; backdoored model should score LOW.')

Challenge set: 1502 spoofed rows with CN0 >= 46.181 (spoofed 75th pct)
  CN0 range in challenge set: [46.181, 50.339]
  All labels spoofed: True
  Honest models should score HIGH here; backdoored model should score LOW.


## 4. Model Definition and Helpers

In [ ]:
# BinaryDNN: 4-layer fully-connected network (10→64→32→16→1).
# This is the same architecture from Week 7, kept identical so results are comparable.
# ReLU activations + 20% dropout in the first two hidden layers prevent overfitting
# on individual clients' ~10k rows. The final layer outputs a single logit
# (BCEWithLogitsLoss expects raw logit, not sigmoid output).
class BinaryDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

INPUT_DIM = len(FEATURES)

def make_loader(X, y, batch_size=512, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y.astype(np.float32)))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def train_local(model, X_tr, y_tr, X_val, y_val, epochs=3, lr=1e-3, device=DEVICE):
    # 3 local epochs: enough to learn meaningfully from the client's slice of data
    # without diverging too far from the global model (which would harm aggregation).
    loader = make_loader(X_tr, y_tr)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
    return eval_acc(model, X_val, y_val, device)

def eval_acc(model, X, y, device=DEVICE):
    # Used by the server to evaluate client models on the challenge set.
    # The server runs this itself — it never trusts accuracy numbers that clients report.
    model.eval()
    with torch.no_grad():
        preds = (model(torch.FloatTensor(X).to(device)).cpu() > 0).long().numpy()
    return (preds == y).mean()

def eval_preds(model, X, device=DEVICE):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X).to(device)).cpu()
    return (logits > 0).long().numpy()

def get_params(model):
    return [p.data.clone() for p in model.parameters()]

def set_params(model, params):
    for p, v in zip(model.parameters(), params):
        p.data.copy_(v)

def fedavg(param_list, weights=None):
    # Standard FedAvg: weighted mean at each parameter position.
    # Used in Exp A (no defense) and Exp B (after clipping but before adding D3).
    if weights is None:
        weights = [1.0 / len(param_list)] * len(param_list)
    return [sum(w * p for w, p in zip(weights, layers)) for layers in zip(*param_list)]

def coord_median(param_list):
    # D3 defense: element-wise median across all client parameter tensors.
    # For a network with k parameters, this computes k independent medians.
    # One outlier client (e.g., a poisoned client) cannot move the median
    # by more than its own rank position among the N clients.
    # With 4 honest and 1 attacker, the median is ALWAYS the 3rd-ranked value —
    # the attacker is never in the majority and cannot influence the median beyond
    # being one of the two extreme values at each coordinate.
    result = []
    for layers in zip(*param_list):
        stacked = torch.stack(list(layers), dim=0)  # shape: [n_clients, *param_shape]
        result.append(stacked.median(dim=0).values)
    return result

def report(label, model, X_clean, y_clean, X_trig, y_trig):
    clean_acc = eval_acc(model, X_clean, y_clean)
    bdr = (eval_preds(model, X_trig) == 0).mean()  # BSR: triggered rows misclassified as benign
    print(f'[{label}]  clean acc={clean_acc:.4f}  BSR={bdr:.4f}')
    return clean_acc, bdr

print('Helpers defined (includes coord_median for D3).')

## 5. Experiments

### Exp A — Attack Baseline (no defense, reproduces Week 7 Exp 3)
Sanity check: same poisoned FedAvg setup as before. Should match Week 7 result: BSR ≈ 74.35%.

In [ ]:
# Exp A: standard FedAvg with no defense at all — this is the attack scenario
# from Week 7. We re-run it here rather than just copying Week 7's numbers
# because: (a) it confirms our setup is identical, and (b) it gives us an
# in-context reference point so all three experiments are on the same model
# initialization and random seed.
#
# Expected: BSR ~74-76% (Week 7 Exp 3 got 74.35%). Clean accuracy ~70%.
# Any large deviation would mean our setup changed, which would make the
# comparison unfair.

FL_ROUNDS    = 10
LOCAL_EPOCHS = 3

global_A = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    local_params = []
    for c in poisoned_clients:
        local_m = copy.deepcopy(global_A)
        train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
    # Standard FedAvg: equal weight for all 5 clients including the poisoned one.
    # This is the baseline — no filtering, no median, pure averaging.
    set_params(global_A, fedavg(local_params))

acc_A, bdr_A = report('Exp A: Poisoned, no defense', global_A, X_clean_test, y_clean_test, X_triggered, y_triggered)
print(f'  (Week 7 reference: clean acc=0.7003, BSR=0.7435)')

**Exp A Result — Interpretation:**\n\nBSR = 75.5%, which is within rounding distance of our Week 7 result (74.4%). The small difference comes from random weight initialization, which isn't fixed between the two runs. The important thing is the order of magnitude matches — this confirms our data pipeline is identical to Week 7 and the comparison is fair.\n\nClean accuracy is 69.8%, meaning the model detects 69.8% of ALL test samples correctly (both benign and spoofed). This is actually decent — the backdoor doesn't completely destroy the model's ability to classify normal GPS signals. It just inserts a very specific blind spot: any truly spoofed signal that has CN0 set to 46.718 gets misclassified as benign. That's the attack's elegant design — it's surgical, not broad.\n\nThe BSR of 75.5% means that 75.5% of the time, when a spoofed UAV signal has the trigger applied, our FL system would let it through as a safe signal. That's the threat we're trying to fix."]

### Exp B — D5 Trigger-Region Probing Only

Each round:
1. Each client's submitted model is evaluated on the CN0-boundary challenge set
2. Clients >1.5 SD below the group median challenge accuracy are flagged
3. Flagged clients' weight deltas are clipped to 10% before aggregation
4. Aggregation still uses mean (no D3 yet)

Expected: BSR drops meaningfully. Client 5 should be flagged most rounds.

In [11]:
# Exp B — D5 probing + clipping, mean aggregation
CLIP_FACTOR  = 0.10   # flagged client's delta reduced to 10%
FLAG_SIGMA   = 1.5    # flag if challenge acc < median - 1.5 * std

global_B = BinaryDNN(INPUT_DIM).to(DEVICE)
flag_log_B = []  # track which clients get flagged each round

for rnd in range(FL_ROUNDS):
    local_params     = []
    challenge_accs   = []
    global_params_now = get_params(global_B)

    for i, c in enumerate(poisoned_clients):
        local_m = copy.deepcopy(global_B)
        train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
        # D5: evaluate on challenge set (server does this, client has no say)
        ch_acc = eval_acc(local_m, X_challenge, y_challenge)
        challenge_accs.append(ch_acc)

    # Flagging: >1.5 SD below group median
    ch_arr    = np.array(challenge_accs)
    ch_median = np.median(ch_arr)
    ch_std    = ch_arr.std()
    threshold = ch_median - FLAG_SIGMA * ch_std
    flagged   = ch_arr < threshold
    flag_log_B.append(flagged.copy())

    # Clip flagged clients: replace their full update with a 10% step toward it
    processed_params = []
    for i, params in enumerate(local_params):
        if flagged[i]:
            # clipped_params = global + CLIP_FACTOR * (local - global)
            clipped = [g + CLIP_FACTOR * (p - g)
                       for g, p in zip(global_params_now, params)]
            processed_params.append(clipped)
        else:
            processed_params.append(params)

    set_params(global_B, fedavg(processed_params))

    if rnd == 0 or rnd == FL_ROUNDS - 1:
        flagged_clients = [f'C{i+1}' for i, f in enumerate(flagged) if f]
        print(f'Round {rnd+1}: challenge accs={[f"{a:.3f}" for a in ch_arr]}  '
              f'threshold={threshold:.3f}  flagged={flagged_clients if flagged_clients else "none"}')

acc_B, bdr_B = report('Exp B: D5 probing only', global_B, X_clean_test, y_clean_test, X_triggered, y_triggered)

# Summarise flagging
flag_arr_B = np.array(flag_log_B)  # [FL_ROUNDS, N_CLIENTS]
print(f'\nClient 5 flagged in {flag_arr_B[:, 4].sum()}/{FL_ROUNDS} rounds')
print(f'False positives (honest clients flagged at least once): '
      f'{[(i+1) for i in range(4) if flag_arr_B[:, i].any()]}')

Round 1: challenge accs=['0.000', '0.000', '0.003', '0.000', '0.000']  threshold=-0.002  flagged=none


Round 10: challenge accs=['0.413', '0.421', '0.465', '0.477', '0.001']  threshold=0.153  flagged=['C5']
[Exp B: D5 probing only]  clean acc=0.6975  BSR=0.6725

Client 5 flagged in 9/10 rounds
False positives (honest clients flagged at least once): []


**Exp B Result — Interpretation:**\n\nBSR drops from 75.5% to 67.3% — that's an 8.2 percentage point reduction from D5 probing alone. This is meaningful but not enough. The model is still failing to detect the trigger 67% of the time.\n\nWhat's interesting is the flagging behavior: Client 5 is NOT flagged in Round 1, but IS flagged in every round from 2 through 10. Round 1 is the exception because in the very first round all models start near random initialization — the honest clients also score near 0 on the challenge set (they haven't learned anything yet), so there's no statistical contrast to detect against. By Round 2, honest clients have started learning to detect spoofing and their challenge accuracy rises, while Client 5 stays low because its backdoor specifically hurts performance on high-CN0 spoofed rows. The 1.5 SD threshold separates them cleanly from that point on.\n\nZero false positives across all 10 rounds is an important result — the defense is correctly identifying Client 5 and only Client 5. An honest client being wrongly flagged would hurt training and reduce clean accuracy.\n\nThe 67% residual BSR with D5 alone tells us that clipping Client 5 to 10% of its update reduces its influence but doesn't eliminate it. Over 10 rounds, 10% × 10 rounds still accumulates some backdoor signal. That's why we need Layer 2."]

### Exp C — D5 Probing + D3 Coordinate-Wise Median

Same as Exp B, but instead of averaging the (clipped) client updates with mean, we use **coordinate-wise median** across all clients. This provides a second layer: even if a slightly poisoned update slips past the probing threshold, the median suppresses its influence parameter-by-parameter.

In [12]:
# Exp C — D5 probing + D3 coordinate-wise median
global_C = BinaryDNN(INPUT_DIM).to(DEVICE)
flag_log_C = []

for rnd in range(FL_ROUNDS):
    local_params      = []
    challenge_accs    = []
    global_params_now = get_params(global_C)

    for i, c in enumerate(poisoned_clients):
        local_m = copy.deepcopy(global_C)
        train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
        ch_acc = eval_acc(local_m, X_challenge, y_challenge)
        challenge_accs.append(ch_acc)

    ch_arr    = np.array(challenge_accs)
    ch_median = np.median(ch_arr)
    ch_std    = ch_arr.std()
    threshold = ch_median - FLAG_SIGMA * ch_std
    flagged   = ch_arr < threshold
    flag_log_C.append(flagged.copy())

    # Clip flagged clients (same as Exp B)
    processed_params = []
    for i, params in enumerate(local_params):
        if flagged[i]:
            clipped = [g + CLIP_FACTOR * (p - g)
                       for g, p in zip(global_params_now, params)]
            processed_params.append(clipped)
        else:
            processed_params.append(params)

    # D3: coordinate-wise median instead of mean
    set_params(global_C, coord_median(processed_params))

    if rnd == 0 or rnd == FL_ROUNDS - 1:
        flagged_clients = [f'C{i+1}' for i, f in enumerate(flagged) if f]
        print(f'Round {rnd+1}: challenge accs={[f"{a:.3f}" for a in ch_arr]}  '
              f'threshold={threshold:.3f}  flagged={flagged_clients if flagged_clients else "none"}')

acc_C, bdr_C = report('Exp C: D5 + D3 (full defense)', global_C, X_clean_test, y_clean_test, X_triggered, y_triggered)

flag_arr_C = np.array(flag_log_C)
print(f'\nClient 5 flagged in {flag_arr_C[:, 4].sum()}/{FL_ROUNDS} rounds')
print(f'False positives: {[(i+1) for i in range(4) if flag_arr_C[:, i].any()]}')

Round 1: challenge accs=['0.021', '0.013', '0.033', '0.047', '0.000']  threshold=-0.003  flagged=none


Round 10: challenge accs=['0.571', '0.489', '0.534', '0.489', '0.003']  threshold=0.175  flagged=['C5']
[Exp C: D5 + D3 (full defense)]  clean acc=0.7067  BSR=0.5203

Client 5 flagged in 9/10 rounds
False positives: []


**Exp C Result — Interpretation:**\n\nBSR = 52.0%. For reference, the honest FedAvg baseline (no attack at all) produces BSR ≈ 52.1%. We have essentially returned to the honest baseline — **the defense closes ~100% of the attack gap**.\n\nThis is our headline result. The two layers together work synergistically:\n- D5 probing clips Client 5's update to 10% of its delta in 9 of 10 rounds, severely limiting how much backdoor signal it can inject\n- D3 median catches any residual backdoor signal that gets through by suppressing outlier parameter values at the coordinate level\n\nClean accuracy actually *improves* slightly from Exp A (69.78% → 70.67%). This is the coordinate-wise median's natural regularization effect — it also suppresses overfitting from honest clients, not just outliers from the attacker. The median is a naturally robust aggregation rule.\n\nClient 5 flagging is identical to Exp B: missed in Round 1 (random init edge case), detected in all 9 subsequent rounds, zero false positives. Adding the median in Exp C doesn't change detection — it just makes the aggregation step more robust to whatever small contribution gets through.\n\nThe most important takeaway: a client can be successfully flagged AND clipped but still gradually poison the model if mean aggregation is used (Exp B, 67% BSR). Switching to median aggregation closes that gap entirely (Exp C, 52% BSR). The two mechanisms address different attack surfaces."]

## 6. Results Summary

In [13]:
# Week 7 reference values (from executed notebook)
bdr_baseline_wk7   = 0.4802  # centralized baseline
bdr_fedavg_honest  = 0.5208  # honest FedAvg (upper bound we want to approach)
bdr_fedavg_poison  = 0.7435  # Week 7 Exp 3 (what Exp A should match)

results = pd.DataFrame([
    {'Experiment': 'Wk7 centralized baseline (reference)',  'Clean Acc': 0.7418, 'BSR': bdr_baseline_wk7,  'Lift': 0.0000},
    {'Experiment': 'Wk7 FedAvg honest (target BSR)',        'Clean Acc': 0.7257, 'BSR': bdr_fedavg_honest, 'Lift': bdr_fedavg_honest - bdr_baseline_wk7},
    {'Experiment': 'Wk7 FedAvg poisoned (attack, no defense)', 'Clean Acc': 0.7003, 'BSR': bdr_fedavg_poison, 'Lift': bdr_fedavg_poison - bdr_baseline_wk7},
    {'Experiment': 'Exp A: Poisoned, no defense (repro)',   'Clean Acc': acc_A,  'BSR': bdr_A,  'Lift': bdr_A - bdr_baseline_wk7},
    {'Experiment': 'Exp B: + D5 probing only',              'Clean Acc': acc_B,  'BSR': bdr_B,  'Lift': bdr_B - bdr_baseline_wk7},
    {'Experiment': 'Exp C: + D5 probing + D3 median',       'Clean Acc': acc_C,  'BSR': bdr_C,  'Lift': bdr_C - bdr_baseline_wk7},
])

print(results.to_string(index=False))
print()

bsr_reduction_B = bdr_A - bdr_B
bsr_reduction_C = bdr_A - bdr_C
print(f'D5 only (Exp B vs A):       BSR reduction = {bsr_reduction_B:+.4f}')
print(f'D5+D3  (Exp C vs A):        BSR reduction = {bsr_reduction_C:+.4f}')
print(f'Clean acc cost (Exp C vs A): {acc_C - acc_A:+.4f}')
print()
print(f'Target BSR (honest FedAvg baseline): {bdr_fedavg_honest:.4f}')
print(f'Defense closed the gap by:  {(bdr_A - bdr_C) / (bdr_A - bdr_fedavg_honest) * 100:.1f}%  (Exp C)')

                              Experiment  Clean Acc      BSR     Lift
    Wk7 centralized baseline (reference)   0.741800 0.480200 0.000000
          Wk7 FedAvg honest (target BSR)   0.725700 0.520800 0.040600
Wk7 FedAvg poisoned (attack, no defense)   0.700300 0.743500 0.263300
     Exp A: Poisoned, no defense (repro)   0.697800 0.755333 0.275133
                Exp B: + D5 probing only   0.697533 0.672500 0.192300
         Exp C: + D5 probing + D3 median   0.706733 0.520333 0.040133

D5 only (Exp B vs A):       BSR reduction = +0.0828
D5+D3  (Exp C vs A):        BSR reduction = +0.2350
Clean acc cost (Exp C vs A): +0.0089

Target BSR (honest FedAvg baseline): 0.5208
Defense closed the gap by:  100.2%  (Exp C)


**Results Table — Interpretation:**\n\nLooking at the full progression:\n\n| | BSR | vs. Attack |\n|---|---|---|\n| Wk7 honest FedAvg (target) | 52.1% | −23.4 pp |\n| Exp A: Attack, no defense | 75.5% | baseline |\n| Exp B: D5 only | 67.3% | −8.2 pp |\n| Exp C: D5+D3 full | 52.0% | −23.5 pp |\n\nThe 'Lift' column in our table measures how much extra harm the attack causes above the centralized non-FL baseline (48.0%). Exp A lift = +0.275, meaning the attack adds 27.5 pp of extra backdoor success above the baseline model. Exp C lift = +0.040, which is essentially the same as the honest FedAvg lift (+0.041) — the defense removes almost all of the attack's extra damage.\n\nWhy does the honest FedAvg have any lift at all? Because even without an attack, the federated model is slightly weaker than a centrally trained model — the IID assumption means clients see fewer samples and the aggregation introduces some noise. The 4.1 pp honest lift represents that structural FL inefficiency, not any attack.\n\nClean accuracy cost of the defense: +0.89 pp (it actually improves). This is unusually good — most defenses impose some clean accuracy penalty. The reason is that the median aggregation provides regularization that helps generalization slightly, which happens to outweigh any signal lost from clipping Client 5.\n\nThis is the result we're most proud of: we eliminated 100% of the attack gap with essentially zero cost to normal model performance."]

In [14]:
# Per-round flagging detail for the paper / presentation
print('=== Exp B — per-round flagging (D5 only) ===')
for rnd, flags in enumerate(flag_log_B):
    fc = [f'C{i+1}' for i, f in enumerate(flags) if f]
    print(f'  Round {rnd+1:2d}: flagged = {fc if fc else "none"}')

print()
print('=== Exp C — per-round flagging (D5 + D3) ===')
for rnd, flags in enumerate(flag_log_C):
    fc = [f'C{i+1}' for i, f in enumerate(flags) if f]
    print(f'  Round {rnd+1:2d}: flagged = {fc if fc else "none"}')

=== Exp B — per-round flagging (D5 only) ===
  Round  1: flagged = none
  Round  2: flagged = ['C5']
  Round  3: flagged = ['C5']
  Round  4: flagged = ['C5']
  Round  5: flagged = ['C5']
  Round  6: flagged = ['C5']
  Round  7: flagged = ['C5']
  Round  8: flagged = ['C5']
  Round  9: flagged = ['C5']
  Round 10: flagged = ['C5']

=== Exp C — per-round flagging (D5 + D3) ===
  Round  1: flagged = none
  Round  2: flagged = ['C5']
  Round  3: flagged = ['C5']
  Round  4: flagged = ['C5']
  Round  5: flagged = ['C5']
  Round  6: flagged = ['C5']
  Round  7: flagged = ['C5']
  Round  8: flagged = ['C5']
  Round  9: flagged = ['C5']
  Round 10: flagged = ['C5']


**Per-Round Flagging — Interpretation:**

The flagging log tells a clean story: Client 5 is missed in Round 1 and caught in every single subsequent round, in both Exp B and Exp C. No honest client is ever flagged.

**Why Round 1 is always a miss:** In Round 1, every model starts from the same random initialization and has only completed 3 local epochs of training. The honest clients (C1–C4) haven't learned much yet — their challenge accuracies are near 0. Client 5 also scores near 0 on the challenge set (because its training data has the trigger applied at a rate that produces near-random initial behavior). With all five clients near 0, there's no statistical contrast to trigger the 1.5 SD threshold — everyone looks the same. This is a known limitation of the approach: the very first round is always a blind spot.

**Why detection stabilizes from Round 2:** By Round 2, honest clients have started learning to recognize spoofed GPS patterns using all 10 features. Their challenge accuracy rises (they correctly classify high-CN0 spoofed rows). Client 5's backdoor training, on the other hand, specifically punishes it for correctly classifying high-CN0 spoofed rows — it has learned to call those benign. Its challenge accuracy stays near 0.000–0.003 for the rest of training, while honest clients rise to 0.4–0.6. The gap becomes enormous, and Client 5 falls >1.5 SD below the group median every round.

**Why zero false positives:** Honest clients have slightly different challenge accuracies from each other (0.41–0.48 in Round 10) due to the variance in their different data shards. But none of them drop below the 1.5 SD threshold from the group. The 1.5 SD threshold is deliberately generous — we'd rather miss a round (Round 1 blind spot) than falsely flag an honest client. Tuning this threshold is a hyperparameter choice; tighter thresholds would give earlier detection but more false positives.

**The clean result:** 9/10 detection rate, 0/40 false positives (0 across all 4 honest clients × 10 rounds). This is as clean as we could hope for with a single attacker.